In [ ]:
import nltk
import string
import pandas as pd
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

#без лемматизации и стемминга
def preprocess_basic(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    return " ".join([t for t in tokens if t not in stop_words and len(t) > 2])

#стемминг
def preprocess_with_stemming(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    stemmed_tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(stemmed_tokens)

#лемматизация
def preprocess_with_lemmatization(text, only_nouns_adj=False):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    lemmatized_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue
        #оставляем только существительные(N) и прилагательные(J)
        if only_nouns_adj:
            if not (tag.startswith('N') or tag.startswith('J')):
                continue
        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    return " ".join(lemmatized_tokens)

dataset = load_dataset("emotion")
train_subset = dataset['train'].select(range(1000))
test_subset = dataset['test'].select(range(300))

y_train = train_subset['label']
y_test = test_subset['label']

results = []

tasks = [
    #Предобработка: без лемм, стемминг + TF-IDF
    ("Без лемм", [preprocess_basic(t) for t in train_subset['text']],
                [preprocess_basic(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    ("Стемминг", [preprocess_with_stemming(t) for t in train_subset['text']],
                [preprocess_with_stemming(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    #Лемматизация + 3 варианта векторов
    ("Лемматизация", [preprocess_with_lemmatization(t) for t in train_subset['text']],
                    [preprocess_with_lemmatization(t) for t in test_subset['text']], [
                        ("Binary", CountVectorizer(binary=True)),
                        ("Frequency", CountVectorizer(binary=False)),
                        ("TF-IDF", TfidfVectorizer())
                    ]),

    #Лемматизация (сущ + прил) + 3 варианта векторов
    ("Лемм (Сущ+Прил)", [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in train_subset['text']],
                        [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in test_subset['text']], [
                            ("Binary", CountVectorizer(binary=True)),
                            ("Frequency", CountVectorizer(binary=False)),
                            ("TF-IDF", TfidfVectorizer())
                        ])
]

for label, tr_txt, ts_txt, vectorizers in tasks:
    for vec_name, vec in vectorizers:
        X_train = vec.fit_transform(tr_txt)
        X_test = vec.transform(ts_txt)

        models_to_test = [
            ("GBM", GradientBoostingClassifier(random_state=42)),
            ("AdaBoost", AdaBoostClassifier(random_state=42))
        ]

        for mod_name, model in models_to_test:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            f1_micro = f1_score(y_test, y_pred, average='micro')
            f1_macro = f1_score(y_test, y_pred, average='macro')
            f1_weighted = f1_score(y_test, y_pred, average='weighted')

            results.append({
                "Task": label,
                "Vector": vec_name,
                "Model": mod_name,
                "F1 Micro": round(f1_micro, 3),
                "F1 Macro": round(f1_macro, 3),
                "F1 Weighted": round(f1_weighted, 3)
            })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

           Task    Vector    Model  F1 Micro  F1 Macro  F1 Weighted
       Без лемм    TF-IDF      GBM     0.703     0.647        0.708
       Без лемм    TF-IDF AdaBoost     0.310     0.086        0.153
       Стемминг    TF-IDF      GBM     0.640     0.592        0.644
       Стемминг    TF-IDF AdaBoost     0.307     0.082        0.151
   Лемматизация    Binary      GBM     0.647     0.602        0.653
   Лемматизация    Binary AdaBoost     0.313     0.086        0.154
   Лемматизация Frequency      GBM     0.643     0.599        0.649
   Лемматизация Frequency AdaBoost     0.313     0.086        0.154
   Лемматизация    TF-IDF      GBM     0.643     0.611        0.649
   Лемматизация    TF-IDF AdaBoost     0.313     0.083        0.154
Лемм (Сущ+Прил)    Binary      GBM     0.543     0.492        0.535
Лемм (Сущ+Прил)    Binary AdaBoost     0.310     0.079        0.148
Лемм (Сущ+Прил) Frequency      GBM     0.547     0.494        0.536
Лемм (Сущ+Прил) Frequency AdaBoost     0.310    

Градиентный бустинг показал лучший результат при Без лемм + TF-IDF, а Адаптивный бустинг показал одинаковые лучшие результаты при Лемматизация + Binary и Лемматизация + Frequency, для дальнейшего сравнения и проверки выберем Binary + лемм для АдаБуст и без лемм + TF-IDF для градиентного

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import f1_score

#GBM без лемм + TF-IDF
texts_train_gbm = [preprocess_basic(t) for t in train_subset['text']]
texts_test_gbm = [preprocess_basic(t) for t in test_subset['text']]
vec_tfidf = TfidfVectorizer()
X_train_gbm = vec_tfidf.fit_transform(texts_train_gbm)
X_test_gbm = vec_tfidf.transform(texts_test_gbm)

#AdaBoost лемм + Binary
texts_train_ada = [preprocess_with_lemmatization(t) for t in train_subset['text']]
texts_test_ada = [preprocess_with_lemmatization(t) for t in test_subset['text']]
vec_bin = CountVectorizer(binary=True)
X_train_ada = vec_bin.fit_transform(texts_train_ada)
X_test_ada = vec_bin.transform(texts_test_ada)

final_tuning = []

for n_est in [100, 250, 400]:
    for depth in [5, 7, 8]:
        gbm = GradientBoostingClassifier(
            learning_rate=0.1,
            n_estimators=n_est,
            max_depth=depth,
            subsample=0.8,
            random_state=42
        )
        gbm.fit(X_train_gbm, y_train)
        y_pred = gbm.predict(X_test_gbm)

        final_tuning.append({
            "Model": "GBM без лемм + TF-IDF",
            "Params": f"n={n_est}, d={depth}",
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

for depth in [10, 20, 30]:
    for n_est in [100, 200, 300]:
        ada = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=depth, random_state=42),
            n_estimators=n_est,
            learning_rate=0.05,
            random_state=42
        )
        ada.fit(X_train_ada, y_train)
        y_pred = ada.predict(X_test_ada)

        final_tuning.append({
            "Model": "AdaBoost лемм + Binary",
            "Params": f"d={depth}, n={n_est}",
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

df_final = pd.DataFrame(final_tuning)
print(df_final.to_string(index=False))

                 Model      Params  F1 Macro  F1 Micro  F1 Weighted
 GBM без лемм + TF-IDF  n=100, d=5     0.649     0.687        0.694
 GBM без лемм + TF-IDF  n=100, d=7     0.675     0.717        0.722
 GBM без лемм + TF-IDF  n=100, d=8     0.663     0.707        0.712
 GBM без лемм + TF-IDF  n=250, d=5     0.656     0.690        0.698
 GBM без лемм + TF-IDF  n=250, d=7     0.678     0.720        0.725
 GBM без лемм + TF-IDF  n=250, d=8     0.676     0.720        0.725
 GBM без лемм + TF-IDF  n=400, d=5     0.666     0.707        0.713
 GBM без лемм + TF-IDF  n=400, d=7     0.681     0.730        0.735
 GBM без лемм + TF-IDF  n=400, d=8     0.668     0.730        0.735
AdaBoost лемм + Binary d=10, n=100     0.124     0.327        0.181
AdaBoost лемм + Binary d=10, n=200     0.143     0.330        0.192
AdaBoost лемм + Binary d=10, n=300     0.142     0.327        0.191
AdaBoost лемм + Binary d=20, n=100     0.253     0.373        0.284
AdaBoost лемм + Binary d=20, n=200     0.281    